# 03 v2 – Braço supervisionado: variantes E0 a E4 (detecção da falha em curso)

**Rótulo:** `1` = minuto dentro de uma falha, `0` = normal (sem horizonte). **Folds por falha** (forward-chaining, embargo de 24 h), modelos Logistic Regression, Random Forest e LightGBM, baseline de acaso por rotação circular.

**Variantes (cumulativas):**
| Variante | O que muda |
|---|---|
| E0 | Leituras de 10 s, sem regras de qualidade, features pontuais (scores agregados por minuto para avaliar) |
| E1 | Silver de 1 min (qualidade, cobertura), features pontuais |
| E2 | + janelas por tempo de 1 h e 6 h (cobertura mínima de 80%) |
| E3 | + tratamento de desbalanceamento (`class_weight`), só no treino |
| E4* | + seleção de features: só analógicos (**exploratória e pós-hoc**: a ideia veio dos resultados sem rótulos) |

**Avaliação nos mesmos minutos** para todas as variantes (os da E2), com os mesmos rótulos. O limiar é escolhido para uma taxa fixa de falsos positivos nos minutos normais do teste: serve para **comparar** modelos e variantes, não é regra de implantação (no braço sem rótulos, o limiar vem da validação).

Resultados descritivos: 4 eventos (3 folds), sem teste de significância.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import config
from detection import in_failure, window_features
from evaluation import circular_shift_null, event_metrics, fixed_fpr_threshold
from splits import forward_chaining_folds

WINDOWS = [60, 360]
MIN_COVERAGE = 0.8
FPR_TARGET = 0.01
EMBARGO = 24
SUSTAINED_RECALL = 0.10
N_TREES = 200
N_SHIFTS = 200
TRAIN_EVAL_ROWS = 100_000
E0_TRAIN_STRIDE = 1          # >1 usa 1 de cada k leituras de 10 s no treino da E0 (mais rápido)
SEED = config.SEED


## 1. Falhas, folds e dados

In [ ]:
rep = pd.read_csv(config.FAILURE_REPORT, parse_dates=["start_time", "end_time"]).sort_values("start_time")
failures = [
    {"id": int(r.failure_id), "start": r.start_time.to_pydatetime(), "end": r.end_time.to_pydatetime()}
    for r in rep.itertuples()
]
fail_start = {f["id"]: pd.Timestamp(f["start"]) for f in failures}
folds = forward_chaining_folds(failures, EMBARGO)

def load_silver(variant):
    d = config.SILVER_DIR / variant
    groups = json.loads((d / "_columns.json").read_text(encoding="utf-8"))
    df = pd.read_parquet(d)
    df["ts"] = pd.to_datetime(df["ts"]).dt.tz_localize(None)
    return df.sort_values("ts").reset_index(drop=True), groups["analog"], groups["digital"]

sil1, analog, digital = load_silver("e1")
sil0, _, _ = load_silver("e0")
sensors = analog + digital

sil1["label"] = in_failure(sil1["ts"], failures)
sil0["label"] = in_failure(sil0["ts"], failures)
label_map = pd.Series(sil1["label"].to_numpy(), index=sil1["ts"])

feat2 = window_features(sil1, analog, digital, WINDOWS, MIN_COVERAGE)
e2 = feat2[feat2["coverage_ok"]].drop(columns="coverage_ok").reset_index(drop=True)
e2["label"] = e2["ts"].map(label_map).astype(int)
e2_feats = [c for c in e2.columns if c not in ("ts", "label")]
e4_feats = [c for c in e2_feats if c not in digital and "_duty_" not in c]

VARIANTS = {
    "E0":  dict(frame=sil0[["ts", "label"] + sensors], feats=sensors, balance=False, minute_agg=True),
    "E1":  dict(frame=sil1[["ts", "label"] + sensors], feats=sensors, balance=False, minute_agg=False),
    "E2":  dict(frame=e2, feats=e2_feats, balance=False, minute_agg=False),
    "E3":  dict(frame=e2, feats=e2_feats, balance=True, minute_agg=False),
    "E4*": dict(frame=e2, feats=e4_feats, balance=True, minute_agg=False),
}
for k, v in VARIANTS.items():
    assert not v["frame"][v["feats"]].isna().any().any(), f"{k}: nulos nas features"
    print(f"{k:<4} {len(v['frame']):>9,} linhas | {len(v['feats']):>3} features | balanceamento: {v['balance']}")

common_ts = set(e2["ts"])
print("Minutos comuns de avaliação:", len(common_ts))
for f in folds:
    print(f.name, "| treino até", f.train_end, "| teste de", f.test_start, "até", f.test_end)

## 2. Modelos e avaliação por fold

In [ ]:
def make_models(balance):
    cw = "balanced" if balance else None
    return {
        "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight=cw)),
        "RandomForest": RandomForestClassifier(
            n_estimators=N_TREES, max_depth=14, min_samples_leaf=20,
            class_weight="balanced_subsample" if cw else None, n_jobs=-1, random_state=SEED),
        "LightGBM": LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=31, class_weight=cw,
            random_state=SEED, verbose=-1, n_jobs=-1),
    }

def eval_frame(variant, ts, score, minute_agg):
    t = pd.DataFrame({"ts": ts.to_numpy(), "score": score})
    if minute_agg:
        t["ts"] = t["ts"].dt.floor("min")
        t = t.groupby("ts", as_index=False)["score"].mean()
    t = t[t["ts"].isin(common_ts)].copy()
    t["label"] = t["ts"].map(label_map).astype(int)
    return t.sort_values("ts").reset_index(drop=True)

rows = []
for variant, spec in VARIANTS.items():
    frame, feats = spec["frame"], spec["feats"]
    for fold in folds:
        train = frame[frame["ts"] <= fold.train_end]
        test = frame[(frame["ts"] > fold.test_start) & (frame["ts"] <= fold.test_end)]
        if variant == "E0" and E0_TRAIN_STRIDE > 1:
            train = train.iloc[::E0_TRAIN_STRIDE]
        if train["label"].sum() == 0 or test.empty:
            print(f"{variant} {fold.name}: sem positivos no treino ou teste vazio; pulando"); continue
        for mname, model in make_models(spec["balance"]).items():
            model.fit(train[feats], train["label"])
            t = eval_frame(variant, test["ts"], model.predict_proba(test[feats])[:, 1], spec["minute_agg"])
            if t["label"].sum() == 0:
                print(f"{variant} {fold.name}: sem minutos de falha na avaliação comum; pulando"); break
            thr = fixed_fpr_threshold(t["score"], t["label"], FPR_TARGET)
            m = event_metrics(t, "score", thr, fail_start[fold.test_failure_id])
            pr, prev = average_precision_score(t["label"], t["score"]), t["label"].mean()
            tr = train.sample(min(len(train), TRAIN_EVAL_ROWS), random_state=SEED)
            pr_train = average_precision_score(tr["label"], model.predict_proba(tr[feats])[:, 1])
            null = circular_shift_null(t, "score", FPR_TARGET, n_shifts=N_SHIFTS, seed=SEED)
            alarmed = t["score"] >= thr
            recall = alarmed[t["label"] == 1].mean()
            rows.append({
                "variante": variant, "modelo": mname, "falha_testada": fold.test_failure_id,
                "minutos_teste": len(t), "minutos_em_falha": int(t["label"].sum()), "prevalencia": prev,
                "pr_auc": pr, "lift": pr / prev, "pr_auc_treino": pr_train,
                "p_pr_auc": (null["pr_auc"] >= pr).mean(),
                "recall_nos_minutos_de_falha": recall,
                "sustentada": float(recall >= SUSTAINED_RECALL),
                "detectada": m["detected"], "deteccao_por_acaso": null["detected"].mean(),
                "atraso_h": -m["lead_h"] if not np.isnan(m["lead_h"]) else np.nan,
                "false_alarms_per_day": m["false_alarms_per_day"], "limiar": thr,
            })
        print(variant, fold.name, "concluído")

res = pd.DataFrame(rows)
res.round(3)

## 3. Resultados

Cuidados de leitura (os mesmos dos notebooks 02 e 04): compare a PR-AUC com a prevalência do fold (**lift**); `detectada` conta qualquer alarme na falha, e `sustentada` exige alarme em pelo menos 10% dos minutos; compare as detecções com `deteccao_por_acaso`; `pr_auc_treino` alta com teste baixo indica memorização. A E4* é exploratória.

In [ ]:
per_fold = res.pivot_table(index=["variante", "modelo"], columns="falha_testada", values="lift").round(1)
per_fold.columns = [f"lift_falha_{c}" for c in per_fold.columns]
per_fold

In [ ]:
summary = (
    res.groupby(["variante", "modelo"])
    .agg(
        falhas_testadas=("falha_testada", "nunique"),
        falhas_detectadas=("detectada", "sum"),
        falhas_sustentadas=("sustentada", "sum"),
        deteccoes_esperadas_por_acaso=("deteccao_por_acaso", "sum"),
        lift_medio=("lift", "mean"),
        recall_minutos_falha=("recall_nos_minutos_de_falha", "mean"),
        atraso_h=("atraso_h", "mean"),
        pr_auc_treino=("pr_auc_treino", "mean"),
        falsos_alarmes_dia=("false_alarms_per_day", "mean"),
    )
    .round(3)
)
summary

In [ ]:
out = config.DOCS_DIR / "resultados_supervisionado_e0_e4.csv"
config.DOCS_DIR.mkdir(exist_ok=True)
res.to_csv(out, index=False)
summary.to_csv(config.DOCS_DIR / "resultados_supervisionado_e0_e4_resumo.csv")
print("Resultados salvos em", out)

## 4. Anotações para o artigo (preencher)

- O balanceamento (E3) melhora o recall? O que custa em falsos alarmes?
- Os só analógicos (E4*) melhoram o ranking também no braço supervisionado?
- E0 (10 s) × E1 (1 min): a agregação ajuda ou atrapalha?
- Janelas (E2) × pontuais (E1): atraso e recall (H1)?
- Falha 2: depende do modelo (treino com uma só falha)?
- Lembrar: 3 folds, E4* exploratória, limiar escolhido no teste (só comparação).